In [ ]:
import os, sys
sys.path.append('src')
import ee, geemap

ee.Authenticate()
ee.Initialize(project='g4bproject')

from getSentinel2 import getSentinel
from functions import prepareSTMs
from Auxiliary import addManagement, addAuxiliary

In [ ]:
#variables derived from EDA.ipynb
variables = ['aspect', 'nightlight', 'clim_b12', 'clim_cmi', 'solar', 'clim_gd5', 'Spring_NDVI_stdDev', 'grassland_prox90m', 'Spring_NDMI_stdDev', 'tpi', 'grassland_prox500m', 'elevation', 'grassland_prox360m', 'cropland_prox500m', 'SmTno_090m', 'Autumn_EVI_stdDev', 'Autumn_SWIR_ratio_median', 'smTno_045m', 'R3_p25', 'forest_prox500m', 'nir_stdDev', 'swir1_p90', 'nir_p25', 'Spring_Summer_SWIR_ratio_median', 'Carpathians_2020_Mowing_Intensity_DOY1', 'Spring_Summer_EVI_stdDev', 'blue_p10', 'Spring_EVI_median', 'bgTno_090m', 'Spring_EVI_stdDev', 'Autumn_EVI_median', 'Carpathians_2019_Mowing_Intensity_DOY1', 'Summer_EVI_stdDev', 'Summer_EVI_median', 'swir1_p10', 'blue_p75', 'Spring_Summer_EVI_median', 'swir2_p90', 'clim_gf5', 'Carpathians_2024_Mowing_Intensity_DOY1', 'Summer_SWIR_ratio_median', 'Autumn_MSAVI_stdDev', 'SummerAutumn_EVI_stdDev', 'swir2_p75', 'nir_median', 'Summer_SWIR_ratio_stdDev', 'forest_prox360m', 'swir1_p75', 'red_p10', 'Spring_MSAVI_stdDev']

In [27]:
aoi = (ee.FeatureCollection('projects/g4bproject/assets/Carpathians'))
grid = (ee.FeatureCollection("projects/g4bproject/assets/grids/grid90km") #/grid45km and /grid22_5km possible
        .filterBounds(aoi))
grid45 = (ee.FeatureCollection("projects/g4bproject/assets/grids/grid45km") #/grid45km and /grid22_5km possible
        .filterBounds(aoi))

grassMask = ee.Image(f'projects/ee-simonopravil/assets/LULC/Carpathians/RandomForest').select('label').eq(5)
grassMask

In [ ]:
gridFitlered = grid45.filter(ee.Filter.inList('id', [424, 444, 484, 583]))

In [22]:
folder_ = 'projects/g4bproject/assets/evaSam_Carps' #v1 for Carps, v2 for Alps is because the later usage of the code
folder_path = f'{folder_}/'
assets = ee.data.listAssets({'parent': folder_path})
# Extract the asset IDs from the list.
asset_ids = [asset['id'] for asset in assets['assets']]
len(asset_ids)

fcol = None
for asset in asset_ids:
    f = ee.FeatureCollection(asset)
    if fcol is None:
        fcol = f
    else:
        fcol = fcol.merge(f)

features = fcol.select(variables+['rich_har'])

In [23]:
rf = ee.Classifier.smileRandomForest(numberOfTrees = 300,  seed=42).train(features, classProperty = 'rich_har', inputProperties = variables).setOutputMode('REGRESSION')

In [30]:
start=2022 #define the start year
end=2024 #define the end year

for idd in [424, 444, 484, 583]:
    gridFitlered = grid45.filter(ee.Filter.eq('id', idd))
    s2 = getSentinel(gridFitlered, start, end)
    predictors = prepareSTMs(s2)
    predictors = addAuxiliary(predictors)
    predictors = addManagement(predictors)
    predictors = predictors.select(variables)

    rich = predictors.classify(rf).updateMask(grassMask)
    task = ee.batch.Export.image.toDrive(**{
            'image': rich,
            'description': f'test_{idd}',
            'fileNamePrefix': f'test_{idd}',
            'folder':'geeExport',
            'crs': 'EPSG:3035',
            'region': gridFitlered.geometry(),
            'scale': 10,
            'maxPixels': 1e10
        })
    task.start()
